# Brain Tumor Classification — Xception (Transfer Learning)

Classifies Brain MRI scans into 4 classes: glioma, meningioma, notumor, pituitary

In [ ]:
# ── Colab setup: run this cell first ────────────────────────────────────────
import os

IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    from google.colab import files
    print('Upload your kaggle.json')
    uploaded = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    os.system('pip install -q kaggle')
    os.system('kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p data --unzip -q')
    TRAIN_DIR = 'data/Training'
    TEST_DIR  = 'data/Testing'
else:
    TRAIN_DIR = '../data/brain_tumor_dataset/Training'
    TEST_DIR  = '../data/brain_tumor_dataset/Testing'

print('Train dir:', TRAIN_DIR)
print('Classes:', os.listdir(TRAIN_DIR))

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import Xception
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
)

print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

# ── Config ───────────────────────────────────────────────────────────────────
IMG_SIZE     = 299
BATCH_SIZE   = 32
EPOCHS_HEAD  = 15
EPOCHS_FINE  = 35
NUM_CLASSES  = 4
UNFREEZE_TOP = 30
CLASSES = ['glioma', 'meningioma', 'notumor', 'pituitary']

# ── Data ─────────────────────────────────────────────────────────────────────
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest',
)
val_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    classes=CLASSES, shuffle=True,
)
val_gen = val_datagen.flow_from_directory(
    TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    classes=CLASSES, shuffle=False,
)
print('Class indices:', train_gen.class_indices)

# ── Build model ───────────────────────────────────────────────────────────────
def build_xception_model(num_classes: int) -> keras.Model:
    base = Xception(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inputs  = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dense(512, activation='relu')(x)
    x       = layers.Dropout(0.4)(x)
    x       = layers.Dense(256, activation='relu')(x)
    x       = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model(inputs, outputs, name='xception_brain_tumor')

model = build_xception_model(NUM_CLASSES)
model.summary()

# ── Phase 1: head only ────────────────────────────────────────────────────────
print('\n── Phase 1: head training ──')
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy', metrics=['accuracy'],
)
model.fit(
    train_gen, epochs=EPOCHS_HEAD, validation_data=val_gen,
    callbacks=[
        ModelCheckpoint('xception_phase1_best.keras', monitor='val_accuracy', save_best_only=True, verbose=1),
        EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
        CSVLogger('xception_phase1_log.csv'),
    ],
)

# ── Phase 2: fine-tune ────────────────────────────────────────────────────────
print('\n── Phase 2: fine-tuning ──')
base_model = model.layers[1]
base_model.trainable = True
for layer in base_model.layers[:-UNFREEZE_TOP]:
    layer.trainable = False
print(f'Trainable layers: {sum(1 for l in model.layers if l.trainable)}')

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy', metrics=['accuracy'],
)
model.fit(
    train_gen, epochs=EPOCHS_FINE, validation_data=val_gen,
    callbacks=[
        ModelCheckpoint('xception_final_best.keras', monitor='val_accuracy', save_best_only=True, verbose=1),
        EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7, verbose=1),
        CSVLogger('xception_phase2_log.csv'),
    ],
)

# ── Evaluate & save ───────────────────────────────────────────────────────────
loss, acc = model.evaluate(val_gen, verbose=1)
print(f'Test accuracy: {acc:.4f}  |  Test loss: {loss:.4f}')

model.save('xception_brain_tumor_final.keras')
print('Saved → xception_brain_tumor_final.keras')

In [ ]:
# ── Download the trained model (Colab only) ───────────────────────────────────
if IN_COLAB:
    from google.colab import files
    files.download('xception_brain_tumor_final.keras')